# Day 7

We have optimized, how we can remove the need of using a separate special class like unit for finding out the gradients and updating parameters and instead use a plain vector.

Now we will try to reduce the classes more or say make it more mathematical.

We will do this by collapsing neuron and just using a Layer. For that what we need to change we will look into that

#### Below is the one we optimized in first version

In [2]:
class Neuron:
    
    def __init__(self, n_inputs, activation='ReLU'):
        self.params = [random.uniform(-1, 1) for i in range(n_inputs+1)]
        self.params_grad = [0] * (n_inputs+1)
        self.activation = activation
        self.Q = 0
        self.input_value = None
        self.n = n_inputs + 1 # +1 as we will have n weights and 1 bias, and you will see next that we are adding 1 in inputs as well

    def forward(self, input_layer):
        # we are adding this as bias needs to be added in the self.Q which is equivalent to self.params[-1] * 1
        self.input_value = input_layer + [1.0] 
        self.Q = sum((x * y for x, y in zip(self.params, self.input_value)))
        if self.activation == 'tanh':
            output_value = math.tanh(self.Q)
        elif self.activation == 'ReLU':
            output_value = max(self.Q, 0)
        else:
            output_value = self.Q
        return output_value

    def backward(self, grad_in):
        dfdq = 1
        if self.activation == 'ReLU':
            dfdq = 1 if self.Q > 0 else 0
        elif self.activation == 'tanh':
            dfdq = 1 - math.tanh(self.Q)**2
        dLdQ = grad_in * dfdq
        for i in range(self.n):
            self.params_grad[i] += dLdQ * self.input_value[i]
        # why this -1?? Because we have only n input neurons, n+1 is for bias term we added
        return [dLdQ * w for w in self.params[:-1]] 
        

    def parameters(self):
        return self.params

    def grads(self):
        return self.params_grad


class Layer:
    
    def __init__(self, n_neurons, n_inputs, activation='ReLU'):
        self.neurons = [Neuron(n_inputs,activation) for i in range(n_neurons)]
        self.input_layer_neurons = n_inputs

    def forward(self, input_layer):
        output = [neuron.forward(input_layer) for neuron in self.neurons]
        return output

    def backward(self, grad_in_list):
        # for each neuron there will be a grad 
        # each neuron will output a list of dL/dX -> we will add them all
        dLdX = [0] * self.input_layer_neurons
        for neuron,grad_in in zip(self.neurons, grad_in_list):
            delta = neuron.backward(grad_in)
            dLdX = [dLdXi + delta_i for dLdXi, delta_i in zip(dLdX, delta)]
        return dLdX

    # we are getting list of parameters by refrence and here as well we need to pass by refrence
    def parameters(self):
        parameters = []
        for neuron in self.neurons:
            parameters.append(neuron.parameters())
        return parameters

    def grads(self):
        grads = []
        for neuron in self.neurons:
            grads.append(neuron.grads())
        return grads
        

class MLP:
    
    def __init__(self, layer_list, activation='ReLU'):
        self.layers = [Layer(layer_list[i], layer_list[i-1], activation) for i in range(1,len(layer_list))]
        
    def forward(self, inputs):
        output = inputs
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def backward(self, loss_grad):
        grad_in = loss_grad
        for layer in reversed(self.layers):
            grad_in = layer.backward(grad_in)

    def parameters(self):
        parameters = []
        for layer in self.layers:
            parameters.append(layer.parameters())
        return parameters

    def grads(self):
        grads = []
        for layer in self.layers:
            grads.append(layer.grads())
        return grads

#### Now to remove the neuron evrything which was in neuron needs to be in a layer

## A neuron role 
### What it stores 
#### 1. parameter list
#### 2. parameter gradient list
#### 3. activation function
#### 4. output of summation before using activation function
#### 5. input list
### What it does 
#### 1. forward i.e output 
#### 2. gradient for each parameter
#### 3. gradient for previous layer
#### 4. also return parameter and its gradients list



#### We will put all these things in Layer itself

In [20]:
import random
import math

In [21]:
class Layer:
    
    def __init__(self, n_neurons, n_inputs, activation='ReLU'):
        self.params = [[random.uniform(-1, 1) for _ in range(n_neurons)] for i in range(n_inputs)] 
        # adding for bias
        self.params.append([0] * n_neurons)
        self.param_grad = [[0] * n_neurons for i in range(n_inputs+1)]
        self.activation = activation
        self.Q = [0] * n_neurons
        self.inputs = n_inputs+1 # no. of neurons in input layer
        self.neurons = n_neurons # no. of neurons
        self.input_values = None

    def forward(self, input_layer):
        # Reset if already has some value
        self.Q = [0] * self.neurons
        # layer output will have output length as no. of neurons i.e. self.neurons
        output = [0] * self.neurons
        # this [1.0] is the bias term
        self.input_values = input_layer + [1.0] 
        # we have 2d array of parameters and 1 d array of inputs
        # parameters -> (n_inputs+1, n_neurons), inputs -> (n_inputs+1)
        for j in range(self.neurons):
            for i in range(self.inputs):
                self.Q[j] += self.params[i][j] * self.input_values[i]
        if self.activation == 'tanh':
            output_value =  [math.tanh(Qi) for Qi in self.Q] 
        elif self.activation == 'ReLU':
            output_value = [max(Qi, 0) for Qi in self.Q]
        else:
            output_value = self.Q
        return output_value

    def backward(self, grad_in):
        dfdq = [1] * self.neurons
        if self.activation == 'ReLU':
            dfdq = [1 if Q_i > 0 else 0 for Q_i in self.Q]
        elif self.activation == 'tanh':
            dfdq = [1 - math.tanh(Q_i)**2 for Q_i in self.Q]
        dLdQ = [grad_i * dfdq_i for grad_i, dfdq_i in zip(grad_in, dfdq)]
        for j in range(self.neurons):
            for i in range(self.inputs):
                self.param_grad[i][j] += dLdQ[j] * self.input_values[i]
        dLdX = [0] * (self.inputs - 1)
        for j in range(self.neurons):
            for i in range(self.inputs-1):
                dLdX[i] += dLdQ[j] * self.params[i][j]
        return dLdX

    # we are getting list of parameters by refrence and here as well we need to pass by refrence
    def parameters(self):
        return self.params

    def grads(self):
        return self.param_grad


class MLP:
    
    def __init__(self, layer_list, activation='ReLU'):
        self.layers = [Layer(layer_list[i], layer_list[i-1], activation) for i in range(1,len(layer_list))]
        
    def forward(self, inputs):
        output = inputs
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def backward(self, loss_grad):
        grad_in = loss_grad
        for layer in reversed(self.layers):
            grad_in = layer.backward(grad_in)

    def parameters(self):
        parameters = []
        for layer in self.layers:
            parameters.append(layer.parameters())
        return parameters

    def grads(self):
        grads = []
        for layer in self.layers:
            grads.append(layer.grads())
        return grads

class MSE:
    def __init__(self):
        pass

    def loss(self, y_pred, y_true):
        inputs = len(y_pred)
        ans = 0.0
        grads = []
        for y_predi, y_truei in zip(y_pred,y_true):
            ans += (y_predi-y_truei)**2 
            grads.append(2*(y_predi-y_truei))
        ans /= inputs
        for i in range(inputs):
            grads[i]/=inputs
        return (ans, grads)


class SGD:
    def __init__(self, parameters, grads, learning_rate=0.01):
        self.parameters = parameters
        self.grads = grads
        self.learning_rate = learning_rate

    def step(self):
        for layer_parameters, layer_grads in zip(self.parameters, self.grads):
            for neuron_parameters, neuron_grad in zip(layer_parameters, layer_grads):
                for i in range(len(neuron_parameters)):
                    neuron_parameters[i] -= self.learning_rate * neuron_grad[i]
                    
        
    def zero_grad(self):
        for layer_grads in self.grads:
            for neuron_grads in layer_grads:
                 for i in range(len(neuron_grads)):
                    neuron_grads[i] = 0


In [28]:
inputs = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]
targets = [[0.0], [1.0], [1.0], [0.0]]
nn = MLP([2,4,4,1], 'tanh')
loss_func = MSE()
optimizer = SGD(nn.parameters(), nn.grads(), 0.05)
epochs = 100
# Training
for epoch in range(epochs):
    total_loss = 0
    for input_value, target in zip(inputs, targets):
        output = nn.forward(input_value)
        loss, loss_grad = loss_func.loss(output, target)
        total_loss += loss
        nn.backward(loss_grad)
        optimizer.step()
        optimizer.zero_grad()
    if epoch%10 == 0:
        print(f"epoch {epoch}, total loss: {total_loss}")
output = nn.forward([1.0, 1.0])
print(output)

epoch 0, total loss: 2.1182971540103748
epoch 10, total loss: 2.5776332163263262
epoch 20, total loss: 2.658020584028432
epoch 30, total loss: 2.669628808403778
epoch 40, total loss: 2.671277395454091
epoch 50, total loss: 2.671511034066127
epoch 60, total loss: 2.67154413575863
epoch 70, total loss: 2.6715488253803623
epoch 80, total loss: 2.671549489770049
epoch 90, total loss: 2.671549583895614
[8.411820962006544e-10]


In [27]:
nn.forward([1.0, 0.0])

[0.2799028534617964]

In [24]:
nn.parameters()

[[[-0.08553936285036512,
   -1.5927525491194219,
   0.6645178672617164,
   0.4811600443945332],
  [-0.4682016739240076,
   -1.4939080837235756,
   -0.015265576871471308,
   0.6424042699501429],
  [0.12226102495186814,
   0.25294954553624804,
   0.08070486484924883,
   -0.5640579939969755]],
 [[-0.6272878459830307,
   -0.18256677047470393,
   0.11005201473561448,
   -0.8531343985986372],
  [0.17702356828991742,
   -1.3590896508585517,
   0.028092300338206547,
   1.003198035440941],
  [0.7349447455885605,
   -0.6549491260865349,
   -0.5357897679048766,
   0.7467505556231266],
  [0.23288844568358466,
   -0.5348938499521664,
   0.8205890963715844,
   0.9749642199169799],
  [-0.10632549997064765,
   -0.26908931972435207,
   0.03566336845676814,
   0.06796731739383607]],
 [[-0.44648519669448566],
  [1.3672689874061772],
  [-0.5401865495895349],
  [-1.1252267337787607],
  [0.0049715821256045385]]]